# College selectivity + family income dataset
Builds one row per college by merging:
1. Urban Institute Education Data Portal (IPEDS): admissions, test scores, Pell share, net price by family income
2. Opportunity Insights (Chetty et al.) college mobility data: parent income, mobility rates

Run all cells, then download `college_dataset.csv`.
Sources to cite: Urban Institute Education Data Portal (https://educationdata.urban.org),
NCES IPEDS, and Chetty, Friedman, Saez, Turner, Yagan (2017) "Mobility Report Cards".

In [ ]:
import re
import requests
import pandas as pd

URBAN = "https://educationdata.urban.org/api/v1/college-university"
YEAR = 2021  # check the portal for the latest year available for each endpoint

# Verify this URL on https://opportunityinsights.org/data/ (look for "Mobility Report Cards: College-level data, Table 2")
OI_URL = "https://opportunityinsights.org/wp-content/uploads/2018/04/mrc_table2.csv"

HEADERS = {"User-Agent": "classroom-data-project"}


def get_all(path, **params):
    """Fetch every page from an Urban API endpoint into a DataFrame."""
    url, rows = f"{URBAN}/{path}", []
    while url:
        r = requests.get(url, params=params if url.endswith("/") else None,
                         headers=HEADERS, timeout=120)
        r.raise_for_status()
        j = r.json()
        rows += j["results"]
        url = j.get("next")
    return pd.DataFrame(rows)


def clean_missing(df):
    """Urban codes missing values as negatives (-1, -2, -3). Turn them into NaN."""
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num] < 0)
    return df


def cols_like(df, pattern):
    return [c for c in df.columns if re.search(pattern, c)]

## 1. Directory (names, state, OPEID for linking to Chetty)

In [ ]:
directory = clean_missing(get_all(f"ipeds/directory/{YEAR}/"))
print(directory.columns.tolist())
keep = ["unitid", "inst_name", "state_abbr", "opeid", "inst_control"]
directory = directory[[c for c in keep if c in directory.columns]]

## 2. Admissions counts and test scores
Counts come from `admissions-enrollment` (one row per college per `sex`; we keep the all-students row).
Test scores come from `admissions-requirements`. IPEDS publishes the 25th, 50th (median) and 75th
percentile of SAT/ACT scores, NOT the average, so we keep the median as the "typical" score.

In [ ]:
adm = clean_missing(get_all(f"ipeds/admissions-enrollment/{YEAR}/"))
print(adm.columns.tolist())
# One row per college per sex (male, female, total...). The all-students row has the most applicants.
adm = adm.sort_values("number_applied").groupby("unitid").tail(1)
adm = adm.rename(columns={"number_applied": "applicants", "number_admitted": "admitted",
                          "number_enrolled_total": "enrolled"})
adm["admit_rate"] = adm["admitted"] / adm["applicants"]
adm["yield"] = adm["enrolled"] / adm["admitted"]
adm = adm[["unitid", "applicants", "admitted", "enrolled", "admit_rate", "yield"]]

req = clean_missing(get_all(f"ipeds/admissions-requirements/{YEAR}/"))
req = req.rename(columns={
    "sat_crit_read_25_pctl": "sat_ebrw_25", "sat_crit_read_50_pctl": "sat_ebrw_50", "sat_crit_read_75_pctl": "sat_ebrw_75",
    "sat_math_25_pctl": "sat_math_25", "sat_math_50_pctl": "sat_math_50", "sat_math_75_pctl": "sat_math_75",
    "act_composite_25_pctl": "act_composite_25", "act_composite_50_pctl": "act_composite_50",
    "act_composite_75_pctl": "act_composite_75"})
for q in ("25", "50", "75"):
    req[f"sat_total_{q}"] = req[f"sat_ebrw_{q}"] + req[f"sat_math_{q}"]   # sum of section percentiles: an approximation
req = req[["unitid", "open_admissions_policy", "sat_percent_submitting", "act_percent_submitting",
           "sat_ebrw_25", "sat_ebrw_50", "sat_ebrw_75", "sat_math_25", "sat_math_50", "sat_math_75",
           "sat_total_25", "sat_total_50", "sat_total_75",
           "act_composite_25", "act_composite_50", "act_composite_75"]]
adm = adm.merge(req, on="unitid", how="outer")
print(adm[["sat_total_50", "act_composite_50", "sat_percent_submitting"]].notna().sum())  # should NOT be all zeros

## 3. Net price by family income, pivoted to one row per college

In [ ]:
net = clean_missing(get_all(f"ipeds/sfa-grants-and-net-price/{YEAR}/"))
print(net.columns.tolist())
print(net["income_level"].value_counts() if "income_level" in net else "no income_level column")

if "income_level" in net:
    # Check the codebook for what each income_level code means; 1-5 are the brackets,
    # 99 (if present) is typically the all-income total.
    # Keep only the real measurements. The other numeric columns (class_level, fips, ftpt,
    # tuition_type, type_of_aid, ...) are category codes and shouldn't be pivoted.
    wanted = ["net_price", "average_grant", "total_grant",
              "number_of_students", "number_receiving_grants"]
    value_cols = [c for c in wanted if c in net.columns]
    net = net.pivot_table(index="unitid", columns="income_level", values=value_cols)
    net.columns = [f"{v}_inc{int(l)}" for v, l in net.columns]
    net = net.reset_index()

## 4. Share of first-time, full-time students receiving each type of aid (incl. Pell)
`sfa-ftft` has one row per college per type_of_aid (plus other category columns). The `type_of_aid`
codes are in the Education Data Portal codebook (look for the sfa-ftft page). Pell grants is one of them.

In [ ]:
ftft = clean_missing(get_all(f"ipeds/sfa-ftft/{YEAR}/"))
for c in ["ftpt", "class_level", "level_of_study", "degree_seeking", "type_of_aid"]:
    print(c, ftft[c].value_counts().to_dict())

# Sanity check: print every aid type for Harvard, then find the code whose share is around 15-20%.
print(ftft[ftft.unitid == 166027][["type_of_aid", "percent_of_students", "average_amount"]])

In [ ]:
# Set this after confirming the Pell code in the codebook and the Harvard printout above.
PELL_CODE = 4   # <-- CONFIRM

pell = ftft[ftft["type_of_aid"] == PELL_CODE]
# one row per college: if the other category columns still leave several rows, keep the largest group
pell = (pell.sort_values("number_of_students").groupby("unitid").tail(1)
        [["unitid", "number_of_students", "percent_of_students", "average_amount"]]
        .rename(columns={"number_of_students": "pell_students", "percent_of_students": "pell_pct",
                         "average_amount": "pell_avg_amount"}))
ftft = pell
print(ftft["pell_pct"].describe())

## 5. Opportunity Insights (Chetty) mobility data

In [ ]:
import io

# The site rejects pandas' default User-Agent (HTTP 403), so fetch with requests and a browser-style header.
r = requests.get(OI_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
if r.status_code == 200:
    oi = pd.read_csv(io.StringIO(r.text))
else:
    # Fallback: download mrc_table2.csv by hand from https://opportunityinsights.org/data/
    # (Mobility Report Cards > "Baseline Cross-Sectional Estimates of Child and Parent Income
    # Distributions by College"), then drag it into Colab's Files panel (left sidebar).
    print(f"Download failed with HTTP {r.status_code}; trying a manually uploaded copy.")
    oi = pd.read_csv("mrc_table2.csv")

print(oi.columns.tolist())
# Table 2 is one row per college. Useful columns: super_opeid (the link key), name,
# par_median (median parent income), par_top1pc (share of students from the top 1%),
# par_q1 (share from the bottom quintile), k_median (students' own income in their mid-30s),
# mr_kq5_pq1 (mobility rate: bottom quintile to top quintile). Read the codebook for the rest.
oi = oi.add_prefix("oi_").rename(columns={"oi_super_opeid": "super_opeid"})

## 6. Merge
Chetty uses `super_opeid` (a 6-digit id that groups multi-campus systems).
IPEDS `opeid` is 8 digits; the first 6 match for most schools.

In [ ]:
directory["opeid6"] = directory["opeid"].astype(str).str.zfill(8).str[:6].astype(int)

df = (directory
      .merge(adm, on="unitid", how="left")
      .merge(net, on="unitid", how="left")
      .merge(ftft, on="unitid", how="left"))

# Where several campuses share one opeid6, keep the campus with the most applicants.
df = (df.sort_values("applicants", ascending=False)
        .drop_duplicates("opeid6"))

df = df.merge(oi, left_on="opeid6", right_on="super_opeid", how="left")
df = df.drop(columns=["opeid6"])

# --- Fallback: match by exact (normalized) college name when the IDs don't line up ---
# Chetty's super_opeid is their own construction and sometimes differs from the IPEDS OPEID
# (e.g. Harvard is 129 in their file). We only use names that appear exactly once in their file.
def norm_name(s):
    s = str(s).lower().replace("&", "and")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def name_similarity(a, b):
    """Share of words in common (0-1). Low values flag questionable ID matches."""
    stop = {"the", "of", "and"}
    A = {w for w in norm_name(a).split() if w not in stop}
    B = {w for w in norm_name(b).split() if w not in stop}
    return len(A & B) / max(1, min(len(A), len(B)))


oi_cols = list(oi.columns)
oi_by_name = (oi.assign(_n=oi["oi_name"].map(norm_name))
                .drop_duplicates("_n", keep=False).set_index("_n"))
unmatched = df["super_opeid"].isna()
names = df.loc[unmatched, "inst_name"].map(norm_name)
hit = names[names.isin(oi_by_name.index)]
# don't reuse a Chetty row that was already matched by ID
hit = hit[~oi_by_name.loc[hit.values, "super_opeid"].isin(df["super_opeid"]).values]
df["oi_match"] = None
df.loc[df["super_opeid"].notna(), "oi_match"] = "opeid"
df.loc[hit.index, oi_cols] = oi_by_name.loc[hit.values, oi_cols].values
df.loc[hit.index, "oi_match"] = "name"
print("matched by name fallback:", len(hit))

df["oi_name_similarity"] = [name_similarity(a, b) if pd.notna(b) else None
                            for a, b in zip(df["inst_name"], df["oi_name"])]

print(df.shape)
print("rows with Chetty data:", df["super_opeid"].notna().sum())
print("rows with admission data:", df["applicants"].notna().sum())
df.to_csv("college_dataset.csv", index=False)
df.head()

## Quick sanity checks
- Does Harvard (or any familiar school) look right? `df[df.inst_name.str.contains("Harvard", na=False)].T`
- Is admit_rate between 0 and 1 for every row?

In [ ]:
print(df["admit_rate"].describe())